<a href="https://colab.research.google.com/github/TalCordova/RMBA_SemB26_TC_SC/blob/main/Course_Project_Text_Mining_TC_SC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course Project — Text Mining Features

**Research Methods for Business Analytics** | Semester B, 2026 | Coller School of Management, Tel Aviv University

**Authors:**
- Tal Cordova, 203868187
- Saar Cohen, 213499312

---

## Overview

In this notebook we build a text-based sentiment model trained on product reviews. The model outputs the **predicted probability that a customer's review is positive** (rating = 1).

This probability is then used as an additional feature in the main classification model (see the Train Notebook), which decides whether to target a customer with a marketing contact.

**Approach:** Bag-of-Words representation → Chi-squared feature selection → Logistic Regression

**Outputs:**
- `reviews_train_preds.csv` — probability predictions for the training set
- `reviews_rollout_preds.csv` — probability predictions for the rollout set

## 1. Mount Drive and Load Data

The training data (`text_training.csv`) contains ~2,000 stemmed word-frequency columns (one per vocabulary token) and a binary `rating` target (1 = positive review, 0 = negative).

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

file_path_saar = '/content/drive/MyDrive/Research Methods for Business Analytics/text_training.csv'

file_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/TM Exercise/text_training.csv'

file_path = file_path_tal

df = pd.read_csv(file_path)

print('File loaded successfully. Here are the first 5 rows:')
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File loaded successfully. Here are the first 5 rows:


,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick,rating
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
## load local
import pandas as pd

df = pd.read_csv('text_training.csv')
df.head(), df.shape

(   ID  singl  nutti  wait  disappoint  spring  crisp  lol  origin  adult  ...  \
 0   1      0      0     0           0       0      0    0       0      0  ...   
 1   2      0      0     0           0       0      0    0       0      0  ...   
 2   3      0      0     0           0       0      0    0       0      0  ...   
 3   4      0      0     0           1       0      0    0       0      0  ...   
 4   5      0      0     0           1       0      0    0       0      0  ...   
 
    glutenfre  coco  finger  smoothi  junk  finicki  rice  natur  stick  rating  
 0          0     0       0        0     0        0     0      0      0       1  
 1          0     0       0        0     0        0     0      0      0       0  
 2          0     0       0        0     0        0     0      0      0       1  
 3          0     0       0        0     0        0     0      0      0       0  
 4          0     0       0        0     0        0     0      0      0       0  
 
 [5 rows x 2

## 2. Fit the Sentiment Model

We use a two-step sklearn `Pipeline`:

1. **SelectKBest (chi²)** — keeps the 1,500 most discriminative word tokens out of ~2,000. Chi-squared is appropriate here because all inputs are non-negative counts and the target is binary.
2. **Logistic Regression (C=0.5, L2, SAGA solver)** — regularised LR is a strong and interpretable baseline for text classification. SAGA is chosen for its speed on sparse, high-dimensional input. C=0.5 (moderate regularisation) was selected via cross-validation.

In [7]:
import pandas as pd
import numpy as np
from sklearn import set_config
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

set_config(display="text")  # avoid HTML repr crash from non-UTF8 Windows locale

# Reload data if not already in memory
if 'df' not in dir() or df is None:
    print("Reloading training data...")
    file_path = '/content/drive/MyDrive/Research Methods for Business Analytics/text_training.csv'
    df = pd.read_csv(file_path)
    print("Done.")

if 'X' not in dir() or 'y' not in dir():
    print("Recreating X and y...")
    X = df.drop(columns=['ID'])
    y = df['rating']
    print("Done.")

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2


best_lr = Pipeline([
    ('select', SelectKBest(chi2, k=1500)),
    ('clf', LogisticRegression(C=0.5, penalty='l2', solver='saga', max_iter=5000))
])
best_lr.fit(X, y)

c:\Users\talcordova.TAU\PycharmProjects\RMBA_SemB26_TC_SC\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Pipeline(steps=[('select',
                 SelectKBest(k=1500,
                             score_func=<function chi2 at 0x00000134895E82C0>)),
                ('clf',
                 LogisticRegression(C=0.5, max_iter=5000, penalty='l2',
                                    solver='saga'))])

## 3. Generate Predictions

We generate **positive-class probabilities** (not hard labels) so the downstream model can use this as a continuous feature. Row order is preserved — IDs are saved alongside predictions to allow a safe merge.

> **Note:** The `halo` and `safeti` columns are dropped from the features, matching the preprocessing applied during training.

### 3a. Predictions for the Training Set

In [9]:
# Load rollout file
reviews_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_training.csv'
local_reviews_train = 'reviews_training.csv'
df_reviews_train = pd.read_csv(local_reviews_train)

df_reviews_train.head()

,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,believ,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick
0,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,40,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,84,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# Save IDs in original order - DO NOT SORT OR SHUFFLE
reviews_train_ids = df_reviews_train['ID']

# Drop same columns as training
X_reviews_train = df_reviews_train.drop(columns=['ID', 'halo', 'safeti'])

In [ ]:
predictions = best_lr.predict_proba(X_reviews_train)

# The predict_proba method returns probabilities for both classes (negative and positive).
# We are interested in the probability of the positive class (usually the second column).
# Extract the probabilities for the positive class.
positive_predictions = predictions[:, 1]

# Verify record count before saving
if len(positive_predictions) != len(df_reviews_train):
    print(f"ERROR: prediction count ({len(positive_predictions)}) does not match rollout count ({len(df_reviews_train)}). File not saved.")
else:
    output = pd.DataFrame({
        'ID': reviews_train_ids,
        'rating': positive_predictions
    })

    # Tal's file
    output_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_train_preds.csv'
    output_path_local = 'reviews_train_preds.csv'
    output.to_csv(output_path_local, index=False)
    print(f"OK: Tal's file saved - {len(output)} predictions")

    print(output.head(10))

OK: Tal's file saved - 1499 predictions
    ID    rating
0    3  0.200771
1   40  0.359464
2   47  0.981491
3   54  0.006640
4   84  0.962683
5   92  0.696444
6   95  0.298339
7  113  0.262681
8  114  0.006567
9  207  0.001673


### 3b. Predictions for the Rollout Set

In [ ]:
# Load rollout file
reviews_rollout = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_rollout.csv'
df_reviews_rollout = pd.read_csv(reviews_rollout)

df_reviews_rollout.head()

,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,believ,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick
0,20002,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20009,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20026,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,20037,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,20046,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_reviews_rollout.shape

(1501, 2001)

In [ ]:
# Save IDs in original order - DO NOT SORT OR SHUFFLE
reviews_rollout_ids = df_reviews_rollout['ID']

# Drop same columns as training
X_reviews_rollout = df_reviews_rollout.drop(columns=['ID', 'halo', 'safeti'])

In [ ]:
predictions = best_lr.predict_proba(X_reviews_rollout)

# The predict_proba method returns probabilities for both classes (negative and positive).
# We are interested in the probability of the positive class (usually the second column).
# Extract the probabilities for the positive class.
positive_predictions = predictions[:, 1]

# Verify record count before saving
if len(positive_predictions) != len(df_reviews_rollout):
    print(f"ERROR: prediction count ({len(positive_predictions)}) does not match rollout count ({len(df_reviews_rollout)}). File not saved.")
else:
    output = pd.DataFrame({
        'ID': reviews_rollout_ids,
        'rating': positive_predictions
    })

    # Tal's file
    output_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_rollout_preds.csv'
    output_path_local = 'reviews_rollout_preds.csv'

    output.to_csv(output_path_local, index=False)
    print(f"OK: Tal's file saved - {len(output)} predictions")

    print(output.head(10))

OK: Tal's file saved - 1501 predictions
      ID        rating
0  20002  5.132756e-08
1  20009  8.160113e-02
2  20026  7.172129e-01
3  20037  9.841181e-02
4  20046  9.578428e-01
5  20047  8.881992e-01
6  20053  2.876338e-02
7  20069  2.278023e-01
8  20077  6.507048e-01
9  20124  1.208782e-02
